# CSV Анализатор с Claude Agent

Простой агент для анализа CSV таблиц через OpenRouter API

In [ ]:
# Установка зависимостей
!pip install anthropic pandas -q

In [ ]:
import pandas as pd
import json
from anthropic import Anthropic
from google.colab import userdata

# Получаем ключ из Colab Secrets (добавь OPENROUTER_API_KEY в Secrets)
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

# Инициализируем клиент для OpenRouter
client = Anthropic(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
# Инструмент для чтения CSV
def read_csv_tool(file_path: str) -> str:
    """Читает CSV файл и возвращает первые строки и информацию о структуре"""
    try:
        df = pd.read_csv(file_path)
        info = {
            'shape': df.shape,
            'columns': list(df.columns),
            'dtypes': df.dtypes.astype(str).to_dict(),
            'head': df.head(10).to_dict(orient='records'),
            'describe': df.describe().to_dict()
        }
        return json.dumps(info, ensure_ascii=False, indent=2)
    except Exception as e:
        return f"Ошибка чтения файла: {str(e)}"

# Инструмент для выполнения запросов к данным
def query_csv_tool(file_path: str, query: str) -> str:
    """Выполняет pandas запрос к CSV данным"""
    try:
        df = pd.read_csv(file_path)
        result = eval(f"df.{query}")
        if isinstance(result, pd.DataFrame) or isinstance(result, pd.Series):
            return result.to_string()
        return str(result)
    except Exception as e:
        return f"Ошибка выполнения запроса: {str(e)}"

# Определяем инструменты для Claude
tools = [
    {
        "name": "read_csv",
        "description": "Читает CSV файл и возвращает структуру данных, первые строки и статистику",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "Путь к CSV файлу"
                }
            },
            "required": ["file_path"]
        }
    },
    {
        "name": "query_csv",
        "description": "Выполняет pandas операции над CSV данными. Используй pandas синтаксис после 'df.', например: 'groupby(\"column\").mean()' или 'sort_values(\"column\", ascending=False).head(5)'",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "Путь к CSV файлу"
                },
                "query": {
                    "type": "string",
                    "description": "Pandas запрос (без 'df.' в начале)"
                }
            },
            "required": ["file_path", "query"]
        }
    }
]

In [ ]:
def process_tool_call(tool_name: str, tool_input: dict) -> str:
    """Обрабатывает вызов инструмента"""
    if tool_name == "read_csv":
        return read_csv_tool(tool_input["file_path"])
    elif tool_name == "query_csv":
        return query_csv_tool(tool_input["file_path"], tool_input["query"])
    return "Неизвестный инструмент"

def run_agent(user_message: str, csv_file: str, max_iterations: int = 5):
    """Запускает агент для анализа CSV"""
    messages = [
        {
            "role": "user",
            "content": f"Проанализируй CSV файл: {csv_file}\n\nВопрос: {user_message}"
        }
    ]
    
    print(f"🤖 Агент начинает работу...\n")
    
    for iteration in range(max_iterations):
        response = client.messages.create(
            model="anthropic/claude-opus-5-5",
            max_tokens=4096,
            tools=tools,
            messages=messages
        )
        
        print(f"📍 Итерация {iteration + 1}")
        
        # Если агент завершил работу
        if response.stop_reason == "end_turn":
            final_response = next(
                (block.text for block in response.content if hasattr(block, "text")),
                None
            )
            print(f"\n✅ Ответ агента:\n{final_response}")
            return final_response
        
        # Обрабатываем вызовы инструментов
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Вызов: {block.name}({block.input})")
                    result = process_tool_call(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            
            messages.append({"role": "user", "content": tool_results})
    
    return "Достигнут лимит итераций"

## Тестирование агента

Загрузи CSV файл и задай вопрос

In [ ]:
# Пример: загрузи свой CSV или используй тестовый
from google.colab import files

uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f"Загружен файл: {csv_filename}")

In [ ]:
# Запуск агента
result = run_agent(
    user_message="Сделай краткий анализ данных и найди интересные закономерности",
    csv_file=csv_filename
)

In [ ]:
# Другие примеры вопросов:
# run_agent("Какие колонки содержат пропущенные значения?", csv_filename)
# run_agent("Найди топ-5 записей по определенному столбцу", csv_filename)
# run_agent("Посчитай корреляцию между числовыми колонками", csv_filename)